# Translate French, English, and Ngiemboon (guembon)

Ngiemboon (also spelled *ngienboum*, *guembon*) is a Grassfields language spoken in Batcham, in the west of Cameroon.

This notebook follows the same steps as the README: build sentence pairs, train, translate. English goes through French. The source texts are not in the repository.

Open the notebook from the repository root.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "nmt").exists():
    raise SystemExit("Open this notebook from the repository root.")
sys.path.insert(0, str(root))


## 1. Sentence pairs

Builds the pairs from the source files. A verse present on only one side is skipped.


In [ ]:
import json
from nmt.parse_bibles import PROCESSED_DIR, build_corpus

pairs = build_corpus(force=False)
stats = json.loads((PROCESSED_DIR / "stats.json").read_text(encoding="utf-8"))
print(json.dumps(stats, ensure_ascii=False, indent=2))
print()
for row in pairs:
    if row["id"] in {"MAT.1.1", "JHN.3.16"}:
        print(row["ref"])
        print(" FR:", row["fr"])
        print(" NG:", row["ng"])
        print()


## 2. Vocabulary

French and Ngiemboon share one vocabulary. Tone marks stay whole.


In [ ]:
from nmt.train import prepare

all_pairs, train_pairs, val_pairs, tokenizer = prepare(force_extract=False)
print("train verses", len(train_pairs), "val verses", len(val_pairs))
print("vocab", tokenizer.vocab_size)
print(tokenizer.encode("Car Dieu a tant aimé le monde", lang_tag="ng")[:20])
print(tokenizer.decode(tokenizer.encode("Car Dieu a tant aimé le monde", lang_tag="ng")))


## 3. The model

The next cell loads a trained model when one is present.


In [ ]:
from nmt.paths import CHECKPOINTS
from nmt.train import load_model

ckpt = CHECKPOINTS / "fr_ngiemboon.pt"
if ckpt.exists():
    model, tokenizer, device = load_model()
    print("loaded", ckpt, "on", device)
else:
    print("No checkpoint. Run: python -m nmt train")


## 4. Translate

Everyday sentences are less reliable than sentences close to the training texts.


In [ ]:
from nmt.decode import translate_pair

examples_fr = [
    "Généalogie de Jésus-Christ, fils de David, fils d’Abraham",
    "Car Dieu a tant aimé le monde qu’il a donné son Fils unique, afin que quiconque croit en lui ne périsse point, mais qu’il ait la vie éternelle",
    "Je suis le chemin, la vérité, et la vie",
]

for text in examples_fr:
    hyp = translate_pair(model, tokenizer, text, "fr2ng", device)
    print("FR:", text)
    print("NG:", hyp)
    print()


In [ ]:
examples_ng = [
    "Mezíŋ pàmangʉa njʉ̀’ Yěsô Kristo gẅie aa na ngẅɛ fùɔ Ndavíd, éfóo na ngẅɛ Ábalam e mɔɔn",
    "Á nue ntí gie e Ssé la lɔg ńkwoŋ njÿó pú’u la gwɔ́ tà á náa Míŋ we wémɔ’ɔ́, légwɔ́ gie wɛ̌ nyìŋ wɔ́ gẅie à zǎb ntʉ́m wé ná Míŋ we ŋwɛ́, à tě ńdá’a ńkaa ndòon tèmì wɔ́, à ge lá’ gẅiin le gwɔ́ tèmì",
]

for text in examples_ng:
    hyp = translate_pair(model, tokenizer, text, "ng2fr", device)
    print("NG:", text)
    print("FR:", hyp)
    print()


## 5. Another sentence

Change the sentence, then run the cell again.


In [ ]:
from nmt.pivot import translate_between

phrase = "Que la grâce et la paix vous soient données"
direction = "fr2ng"  # or "ng2fr", "en2ng", "ng2en"

text, notes = translate_between(model, tokenizer, phrase, direction, device)
print(text)
for note in notes:
    print("Note:", note)


## 6. Command line

From the repository root, once a checkpoint exists:

```bash
python -m nmt "Bonsoir, comment ça va ?"
python -m nmt --from en "Good evening"
python -m nmt --to fr "má"
python -m nmt --to en "má"
```

The source texts are not included. Segond 1910 is in the public domain. The other texts stay with their rights holders.
